In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%pip install -q optuna scikit-learn pandas pyarrow mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import os

PROJECT_DIR = "/content/drive/MyDrive/repositories/aq_edge"
os.chdir(PROJECT_DIR)

print("Working directory:", os.getcwd())
print(os.listdir("."))

Working directory: /content/drive/MyDrive/repositories/aq_edge
['.env', 'config.yaml', '.git', 'LICENSE', 'README.md', 'aq_edge', 'data_preparation.ipynb', 'data_processing.ipynb', 'environment', 'main.py', 'notebooks', 'data', 'artifacts', '.python-version', 'pyproject.toml', 'requirements.txt', 'src', 'uv.lock', '__init__.py', 'training_model.ipynb', 'howto_uv.txt', 'mlflow_backup', 'colab', 'metrics.png', 'losses.png', 'training_model_colab.ipynb', 'hyperparameter_optuna.py', '.gitignore', 'optuna_search']


In [ ]:
!python colab/mlflow_colab.py

In [5]:
!python hyperparameter_optuna.py --optuna.n_trials 1 --device.use_cuda true

[2026-10-20 18:52:39] INFO: Logging to file: /content/drive/MyDrive/repositories/aq_edge/artifacts/preprocessing_20261020_185239.log
[2026-10-20 18:52:39] INFO: Logging to file: /content/drive/MyDrive/repositories/aq_edge/artifacts/metrics_20261020_185239.log
[2026-10-20 18:52:41] INFO: Logging to file: /content/drive/MyDrive/repositories/aq_edge/artifacts/main_20261020_185241.log
INFO:__main__:Logging to file: /content/drive/MyDrive/repositories/aq_edge/artifacts/main_20261020_185241.log
[2026-10-20 18:52:41] INFO: Starting CO2 forecasting hyperparameter optimization
INFO:__main__:Starting CO2 forecasting hyperparameter optimization
[2026-10-20 18:52:41] INFO: Device: cuda
INFO:__main__:Device: cuda
[2026-10-20 18:52:41] INFO: Station: UTEC
INFO:__main__:Station: UTEC
[2026-10-20 18:52:41] INFO: Features: ['CO2', 'TEM', 'HUM', 'VOC']
INFO:__main__:Features: ['CO2', 'TEM', 'HUM', 'VOC']
[2026-10-20 18:52:41] INFO: Target: CO2
INFO:__main__:Target: CO2
[2026-10-20 18:52:41] INFO: Valid 

In [ ]:
import os
import shutil
import sqlite3

os.makedirs(BACKUP, exist_ok=True)

# Consistent SQLite backup
src = sqlite3.connect(DB)
dst = sqlite3.connect("/content/mlflow_snapshot.db")
with dst:
    src.backup(dst)
src.close()
dst.close()
shutil.copy2("/content/mlflow_snapshot.db", f"{BACKUP}/mlflow.db")

# Preserve experiment artifacts
if os.path.isdir(ARTIFACTS):
    shutil.copytree(
        ARTIFACTS, f"{BACKUP}/mlartifacts", dirs_exist_ok=True
    )

print("MLflow database and artifacts backed up to Drive.")

In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone

import optuna

output_dir = Path("optuna_results")
output_dir.mkdir(parents=True, exist_ok=True)

# Save all trials
study.trials_dataframe().to_csv(
    output_dir / "trials.csv",
    index=False,
)

# Save the best trial
if study.best_trial is not None:
    result = {
        "study_name": study.study_name,
        "best_trial": study.best_trial.number,
        "objective": "validation_r2",
        "best_value": study.best_value,
        "params": study.best_trial.params,
        "saved_at": datetime.now(timezone.utc).isoformat(),
    }

    with (output_dir / "best_params.json").open("w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

print(f"Results saved to {output_dir.resolve()}")